In [1]:
# CO2 LSTM FORECASTER -PER COUNTRY(TRAIN–TEST)

import pandas as pd
import numpy as np #math
import random
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error #for mae calculation
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input


# 1.Load data
file_path="/kaggle/input/climate-and-energy-consumption-dataset-20202024/global_climate_energy_2020_2024.csv"
df = pd.read_csv(file_path) #reads csv as dataframe
df['date']=pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['date'])#converts text dated intro real dates
df['country']=df['country'].astype(str).str.strip().str.title()
df = df.sort_values(['country', 'date'])#sorts country and date
# 2.Features
FEATURES = [
    'co2_emission',
    'industrial_activity_index',
    'energy_consumption',
    'avg_temperature',
    'energy_price'
]

df['co2_next_day']=df.groupby('country')['co2_emission'].shift(-1)

df = df[df.groupby('country')['co2_emission']
          .transform('count') - df.groupby('country').cumcount() > 1]

for col in FEATURES +['co2_next_day']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    
for country in df['country'].unique():#loop every country
    country_mask = df['country'] == country#country row
    for col in FEATURES:#clean all feature rows
        rolling_mean = (
            df.loc[country_mask, col]
            .rolling(3, center=True)#day 3 uses da1,2,2 and without day2,3,4
            .mean() )

        rolling_std = (
            df.loc[country_mask, col]
            .rolling(3, center=True)
            .std() )
#if value-mean>2*std=True, so it is noise
        noisy = (abs(df.loc[country_mask, col] - rolling_mean) > 2 * rolling_std ) 
#replace noisy values with the mean
#np.where(condition, value_if_true, value_if false
        df.loc[country_mask, col] = np.where(
            noisy,
            rolling_mean,
            df.loc[country_mask, col]#keep original one
        )


for col in FEATURES:
    df[col] = df[col].fillna(
        df[col].rolling(3, min_periods=1).mean()
    )
df = df.dropna(subset=FEATURES + ['co2_next_day'])

# 3. Create Sequences
WINDOW = 9

def create_sequences(df_country):
    if len(df_country) < WINDOW + 10:
        return None, None, None, None

    scaler_x = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_data = scaler_x.fit_transform(df_country[FEATURES])#changes features into 0-1(CO2 500 becomes 0,45)
    y_data = scaler_y.fit_transform(df_country[['co2_next_day']])

    X, y = [], []
    for i in range(len(X_data) - WINDOW):
        X.append(X_data[i:i +WINDOW])
        y.append(y_data[i+ WINDOW])

    return np.array(X), np.array(y), scaler_x, scaler_y

# 4.TRAIN–TEST Data
country_models ={}
country_histories ={}
country_names=[]

print("Preparing countries...")

for country in sorted(df['country'].unique()):
    df_c = df[df['country']== country].copy()#gets one country

    X, y, scaler_x, scaler_y = create_sequences(df_c)

    if X is None or len(X) < 50:
        print(f"{country}: skipped (insufficient data)")
        continue

    train_size = int(len(X) * 0.8)

    country_models[country] = {
        'X_train': X[:train_size],
        'y_train': y[:train_size],
        'X_test': X[train_size:],
        'y_test': y[train_size:],
        'scaler_x': scaler_x,
        'scaler_y': scaler_y,
        'df': df_c
    }

    country_names.append(country)
    print(f"{country}: Train={train_size}, Test={len(X)-train_size}")

# 5. Model
def build_lstm():
    model = Sequential([
        #input: 9 days x 5 features
        Input(shape=(WINDOW, len(FEATURES))),
        #LSTM learns longer patterns
         LSTM(24, return_sequences=True),
        Dropout(0.2),

        LSTM(12, return_sequences=False),
        Dropout(0.2),

        Dense(16, activation='relu'),
        Dropout(0.2),

        Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.0003
        ),
        loss='mse',
        metrics=['mae'])
    return model

# 6.Train
print("\nTraining models...\n")

train_accuracies = [] 
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,#if val_loss doesnt improve for 10 epoches stop training early
    restore_best_weights=True#go back to best model weights
)

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,#reduce learning rate by 50%
    patience=5,#wait 5 epoches without improvment before reducing LR
    min_lr=1e-6#never reduce training below this
)
for country in country_names:
    print(f"Training {country}")

    model = build_lstm()
    data = country_models[country]

    history = model.fit(
    data['X_train'],
    data['y_train'],
    validation_split=0.2,
    epochs=200,
    batch_size=16,
    verbose=1,
    callbacks=[early_stop, lr_reduce]
)
    country_models[country]['model'] = model
    country_histories[country] = history
    
    y_train_pred = model.predict(data['X_train'], verbose=0)#predicts train data
    y_train_pred_real = data['scaler_y'].inverse_transform(y_train_pred)
    y_train_real = data['scaler_y'].inverse_transform(data['y_train'])
    
    mae_train = mean_absolute_error(y_train_real, y_train_pred_real)
    mean_actual_train = np.mean(y_train_real)
    accuracy_train = max(0, 100 - (mae_train / mean_actual_train * 100))
    train_accuracies.append(accuracy_train)
    
    print(f"{country} Training Accuracy: {accuracy_train:.1f}%")

print(f"\nAverage Training Accuracy: {np.mean(train_accuracies):.1f}%") 

# 7. Testing
print("TEST RESULTS")
print("="*40)
accuracies=[]
for country in country_names:
    data=country_models[country]
    model=data['model']
    #Predict CO2 values from unseen test data
    y_pred=model.predict(data['X_test'],verbose=0)

    #convert predictions back from 0-1 scale to real CO2 values
    y_pred_real=data['scaler_y'].inverse_transform(y_pred)
    y_test_real=data['scaler_y'].inverse_transform(data['y_test'])

    #Calculate average prediction error
    mae=mean_absolute_error(y_test_real,y_pred_real)

    #Average real CO2 value (used to calculate accuracy)
    mean_actual=np.mean(y_test_real)

    #Convert error into percentage accuracy
    accuracy=max(0,100-(mae/mean_actual*100))
    accuracies.append(accuracy)
    print(f"{country:<20} MAE: {mae:>8.2f} | Accuracy: {accuracy:>6.1f}%")
#Average accuracy across all countries
print("\nAverage Test Accuracy:",np.mean(accuracies),"%")


# 8.One window test output (per country)

for country in country_names:
    data=country_models[country]
    df_c=data['df']
    model=data['model']
    X_test=data['X_test']
    y_test=data['y_test']
    # Skip if no test data
    if len(X_test)==0:
        continue
    #last test window
    i=-1
    
    y_pred=model.predict(X_test,verbose=0)
    # Convert scaled predictions back to real values
    y_pred_real=data['scaler_y'].inverse_transform(y_pred).flatten()
    y_test_real=data['scaler_y'].inverse_transform(y_test).flatten()
    #Find the dates used for this prediction window
    total_sequences=len(X_test)+len(data['X_train'])
    start_idx=len(df_c)-total_sequences-1+i
    #Get the 9 days used as input
    window_df=df_c.iloc[start_idx:start_idx+WINDOW]
    #Get the actual day 10 row
    target_row=df_c.iloc[start_idx+WINDOW]
    #Calculate prediction error
    error=abs(y_pred_real[i]-y_test_real[i])
    #Convert error into percentage
    error_pct=error/y_test_real[i]*100
    #Accuracy for this single prediction
    accuracy=max(0,100-error_pct)
    print(f"\n--- {country} ---")

    print(f"Window dates: {window_df['date'].iloc[0].date()} to {window_df['date'].iloc[-1].date()}")
    print("Day | Date       | CO2    | Industrial | Energy | Temp  | Price")
    print("-"*75)
    
    #Print the 9 input days
    for d,row in enumerate(window_df.itertuples(),start=1):

        print(f"{d:>3} | {row.date.date()} | {row.co2_emission:>6.1f} | "
              f"{row.industrial_activity_index:>10.2f} | {row.energy_consumption:>6.1f} | "
              f"{row.avg_temperature:>5.1f} | {row.energy_price:>6.1f}")
    #Show prediction vs real value
    print(f"Predicted day 10 CO2: {y_pred_real[i]:.2f}")
    print(f"Actual day 10 CO2:    {y_test_real[i]:.2f}")
    print(f"Error: {error:.2f} ({error_pct:.1f}%)")
    print(f"Accuracy: {accuracy:.1f}%")

2026-06-13 10:17:16.745479: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781345836.978236      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781345837.045375      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781345837.571288      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781345837.571356      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781345837.571359      16 computation_placer.cc:177] computation placer alr

Preparing countries...
Australia: Train=568, Test=142
Brazil: Train=568, Test=142
Canada: Train=568, Test=142
China: Train=568, Test=142
France: Train=568, Test=142
Germany: Train=568, Test=142
India: Train=568, Test=142
Indonesia: Train=568, Test=142
Italy: Train=568, Test=142
Japan: Train=568, Test=142
Mexico: Train=568, Test=142
Netherlands: Train=568, Test=142
Norway: Train=568, Test=142
Poland: Train=568, Test=142
South Africa: Train=568, Test=142
Spain: Train=568, Test=142
Sweden: Train=568, Test=142
Turkey: Train=568, Test=142
United Kingdom: Train=568, Test=142
United States: Train=568, Test=142

Training models...

Training Australia
Epoch 1/200


2026-06-13 10:17:31.700732: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - loss: 0.2154 - mae: 0.3862 - val_loss: 0.1240 - val_mae: 0.2760 - learning_rate: 3.0000e-04
Epoch 2/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0963 - mae: 0.2510 - val_loss: 0.0749 - val_mae: 0.2286 - learning_rate: 3.0000e-04
Epoch 3/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0905 - mae: 0.2480 - val_loss: 0.0770 - val_mae: 0.2295 - learning_rate: 3.0000e-04
Epoch 4/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0797 - mae: 0.2359 - val_loss: 0.0758 - val_mae: 0.2290 - learning_rate: 3.0000e-04
Epoch 5/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0889 - mae: 0.2477 - val_loss: 0.0756 - val_mae: 0.2290 - learning_rate: 3.0000e-04
Epoch 6/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0816 - mae: 0.2392 - val_loss: 0.0755 - val_mae: 0.2289 - learning_rate: 3.0000e-04
Epoch 7/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0970 - mae: 0.2598 - val_loss: 0.0751 - val_mae: 0.2287 - learning_rate: 3.00